# Living near the tracks: does station proximity reduce car ownership, and does rail reliability matter?

**TIL6022 Python Programming – Project Proposal**

**Group:** [group number]

**Authors:** [Name 1], [Name 2], [Name 3], [Name 4], [Name 5]

### Author contribution statement

All authors contributed equally to formulating the research questions, selecting and checking the datasets, and writing this proposal.

## Introduction

The Netherlands faces a large housing shortage, and national policy increasingly steers new construction towards locations near railway stations. The underlying assumption is simple: people who live close to a station need a car less often. Fewer cars per household means less congestion, lower emissions and less space lost to parking.

That assumption is rarely tested at a national scale. It is also incomplete. A station only offers a real alternative to the car if the trains actually run. When a line is frequently disrupted, living near its station may do little to reduce car dependency.

This project combines open neighbourhood statistics from CBS with station and disruption data from Rijden de Treinen. We test how strongly car ownership falls with proximity to a station, whether that depends on the type of station, and whether it weakens where rail service is less reliable. The results are relevant for housing and mobility policy: building near stations only pays off if those stations deliver dependable service.

## Objectives

1. Build a reproducible pipeline that links CBS neighbourhood data to Dutch railway stations and their disruption history.
2. Map the spatial distribution of car ownership and station distance across the Netherlands.
3. Estimate the relationship between station distance and car ownership, controlling for income, urbanisation and household composition.
4. Test whether this relationship differs by station type and by station-level disruption exposure.
5. Translate the findings into implications for station-oriented housing policy.

## Research questions

**Main question:** Is there an association between distance to the nearest railway station and household car ownership in Dutch neighbourhoods (2023), and to what extent does this association vary by station type and disruption frequency (2021–2023)? 


**SQ1:** How were average car ownership per household, distance to the nearest station, and train disruptions spread across Dutch neighbourhoods in 2023? 

**SQ2:** What was the baseline between distance to the nearest station and car ownership per household in 2023, when accounting for income, urbanisation, household size, and density? 

**SQ3:** How did the association between station distance and car ownership per household differ between major stations (intercity/junction hubs) and minor local stations in 2023? 

**SQ4:** Was the association between living near a station and car ownership per household weaker in neighbourhoods near stations with frequent train disruptions (2021–2023)? 

**SQ5:** What do these results mean for housing plans near stations (knooppuntontwikkeling) and investments in train reliability in the Netherlands? 


## Geographical and temporal scale

- **Geographical scale:** all Dutch neighbourhoods as defined by CBS in 2023, linked to all Dutch railway stations.
- **Temporal scale:** a cross-sectional analysis with 2023 as reference year. We use the 2023 data so neighbourhood codes match. Disruption exposure is averaged over 2021–2023 to smooth it out. We though that otherwise you do not have enough data if you use just one year. One outlier will then effect your whole outcome.

## Datasets

| Dataset | Source | Key variables |
| --- | --- | --- |
| [Kerncijfers wijken en buurten 2023 (85618NED)](https://www.cbs.nl/nl-nl/cijfers/detail/85618NED) | CBS | Cars per household, income, urbanisation, household size |
| [Nabijheid voorzieningen 2023 (85830NED)](https://www.cbs.nl/nl-nl/cijfers/detail/85830NED) | CBS | Road distance to nearest train station and major transfer station |
| [Wijk- en buurtkaart 2023](https://www.cbs.nl/nl-nl/dossier/nederland-regionaal/geografische-data/wijk-en-buurtkaart-2023) | CBS / Kadaster (PDOK) | Neighbourhood polygons (EPSG:28992) |
| [Railway stations](https://www.rijdendetreinen.nl/en/open-data/stations) | Rijden de Treinen | Station code, station type, coordinates |
| [Train disruptions 2021–2023](https://www.rijdendetreinen.nl/en/open-data/disruptions) | Rijden de Treinen | Affected stations, cause group, duration (minutes) |


## Intended data analysis pipeline

**Step 1:** download the CBS tables via the cbsodata package and the neighbourhood map and Rijden de Treinen files manually from their websites; all raw files are stored in data/raw/. (this step has already been done, see Data availability check)

**Step 2:** we keep only neighbourhood-level rows, exclude neighbourhoods without car ownership figures, and delete outliers such as neighbourhoods where lease companies register large numbers of cars.

**Step 3:** CBS provides the distance to the nearest station, but not which station it is. We therefore link each neighbourhood's middlepoint to its nearest station, adding the station code and type.

**Step 4:** we split the affected station codes per disruption and calculate, per station, the average number of disruptions and disruption minutes per year (2021–2023).

**Step 5:** the station reliability figures are joined onto the neighbourhoods via the station code, completing the analysis table.

**Step 6:** maps of car ownership and station distance, and a scatter plot of distance versus car ownership, to explore whether a pattern exists.

**Step 7:** a linear regression estimates how car ownership changes with station distance, controlling for income and urbanisation to separate the station effect from, for example, wealthy rural areas. Interaction terms test whether this effect is stronger for major stations (SQ3) and weaker for less reliable stations (SQ4).

**Step 8:** the results are translated into policy implications, for example whether building near stations only reduces car ownership when those stations are reliable.

## Data availability check

The cells below load each dataset to confirm that the data is accessible and contains the variables we need. 

In [9]:
from pathlib import Path
import pandas as pd

raw_dir = Path('data/raw')

bestanden = ['kwb_2023.csv', 'nabijheid_2023.csv', 'wijkenbuurten_2023_v3.gpkg',
             'stations-2023-09-nl.csv',
             'disruptions-2021.csv', 'disruptions-2022.csv', 'disruptions-2023.csv']

for f in bestanden:
    print(f'{f:35s}', 'OK' if (raw_dir / f).exists() else 'ONTBREEKT')

kwb_2023.csv                        OK
nabijheid_2023.csv                  OK
wijkenbuurten_2023_v3.gpkg          OK
stations-2023-09-nl.csv             OK
disruptions-2021.csv                OK
disruptions-2022.csv                OK
disruptions-2023.csv                OK


In [10]:
kwb = pd.read_csv(raw_dir / 'kwb_2023.csv', low_memory=False)
print(kwb.shape)
kwb[['Codering_3', 'SoortRegio_2', 'PersonenautoSPerHuishouden_112',
     'GemiddeldInkomenPerInwoner_81', 'MateVanStedelijkheid_125']].head()

(18116, 128)


,Codering_3,SoortRegio_2,PersonenautoSPerHuishouden_112,GemiddeldInkomenPerInwoner_81,MateVanStedelijkheid_125
0,NL00,Land,1.1,32.8,2.0
1,GM1680,Gemeente,1.4,32.8,5.0
2,WK168000,Wijk,1.3,34.1,5.0
3,BU16800000,Buurt,1.3,33.4,5.0
4,BU16800009,Buurt,1.5,49.5,5.0


In [11]:
nabijheid = pd.read_csv(raw_dir / 'nabijheid_2023.csv', low_memory=False)
print(nabijheid.shape)
nabijheid[['Codering_3', 'AfstandTotTreinstationsTotaal_90',
           'AfstandTotBelangrijkOverstapstation_91', 'AfstandTotOpritHoofdverkeersweg_89']].head()

(18116, 116)


,Codering_3,AfstandTotTreinstationsTotaal_90,AfstandTotBelangrijkOverstapstation_91,AfstandTotOpritHoofdverkeersweg_89
0,NL00,5.3,10.8,1.9
1,GM1680,12.4,14.4,1.5
2,WK168000,13.3,14.3,1.5
3,BU16800000,13.2,14.2,1.5
4,BU16800009,13.7,15.3,2.6


In [12]:
stations = pd.read_csv(raw_dir / 'stations-2023-09-nl.csv')
print(stations.shape)
stations[['code', 'name_long', 'type', 'geo_lat', 'geo_lng']].head()

(397, 11)


,code,name_long,type,geo_lat,geo_lng
0,HT,'s-Hertogenbosch,knooppuntIntercitystation,51.690480,5.293620
1,HTO,'s-Hertogenbosch Oost,stoptreinstation,51.700554,5.318333
2,HDE,'t Harde,stoptreinstation,52.409168,5.893611
3,ATN,Aalten,stoptreinstation,51.921327,6.578627
4,AC,Abcoude,stoptreinstation,52.278500,4.977000


In [13]:
storingen = pd.concat(
    [pd.read_csv(raw_dir / f'disruptions-{jaar}.csv') for jaar in [2021, 2022, 2023]],
    ignore_index=True,
)
print(storingen.shape)
storingen[['rdt_station_codes', 'cause_group', 'start_time', 'duration_minutes']].head()

(15541, 14)


,rdt_station_codes,cause_group,start_time,duration_minutes
0,"HGL, HGLO, ODZ",rolling stock,2021-01-01 08:12:55,197.0
1,"ASD, ASDM, ASSP, DMN, WP",rolling stock,2021-01-01 10:17:19,12.0
2,"BRN, DLD, SD, ST, STZ",infrastructure,2021-01-01 10:44:22,421.0
3,"GBG, HDB, MRB",rolling stock,2021-01-01 12:25:19,20.0
4,"BGN, RB",accidents,2021-01-01 15:17:02,42.0


In [15]:
import geopandas as gpd

buurten_kaart = gpd.read_file(raw_dir / 'wijkenbuurten_2023_v3.gpkg', layer='buurten', rows=5)
buurten_kaart[['buurtcode', 'personenautos_per_huishouden',
               'treinstation_gemiddelde_afstand_in_km', 'geometry']].head()

c:\Users\alecm\anaconda3\envs\TIL6022-26\Lib\site-packages\pyogrio\core.py:34: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


,buurtcode,personenautos_per_huishouden,treinstation_gemiddelde_afstand_in_km,geometry
0,BU09989999,-99997.0,-99997.0,"MULTIPOLYGON (((123629.78 379674.57, 123627.21..."
1,BU00349997,-99997.0,-99997.0,"MULTIPOLYGON (((150087.299 479382.379, 150000...."
2,BU00509997,-99997.0,-99997.0,"MULTIPOLYGON (((155047.32 474836.246, 155049.0..."
3,BU00609998,-99997.0,-99997.0,"MULTIPOLYGON (((196000 608000, 195250 607500, ..."
4,BU00729998,-99997.0,-99997.0,"MULTIPOLYGON (((158000 581000, 158331.38 58046..."


## References

- CBS (2026). *Kerncijfers wijken en buurten 2023* (85618NED). StatLine. https://www.cbs.nl/nl-nl/cijfers/detail/85618NED
- CBS (2025). *Nabijheid voorzieningen; afstand locatie, wijk- en buurtcijfers 2023* (85830NED). StatLine. https://www.cbs.nl/nl-nl/cijfers/detail/85830NED
- CBS & Kadaster (2026). *Wijk- en buurtkaart 2023*, version 3. https://www.cbs.nl/nl-nl/dossier/nederland-regionaal/geografische-data/wijk-en-buurtkaart-2023
- Rijden de Treinen (2026). *Open data: railway stations*. CC0. https://www.rijdendetreinen.nl/en/open-data/stations
- Rijden de Treinen (2026). *Open data: train disruptions*. CC BY 4.0. https://www.rijdendetreinen.nl/en/open-data/disruptions
- [Literature on station proximity and car ownership: to be added.]